In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings("ignore")
import pickle
import os

In [26]:
bloodReport = pd.read_csv(r"C:\Users\direk\Disease_risk_predictor_-3\SYSTEM\dataset\blood_report.csv")
bloodReport.head()

,wbc,rbc,hemoglobin,hematocrit,platelets,glucose,creatinine,bun,diabetes_risk,anemia_risk,infection_risk
0,8.25,5.17,16.60,50.58,241.24,90.71,0.52,8.73,19.70,7.33,3.38
1,7.29,5.56,15.89,47.11,267.77,94.68,0.73,5.40,11.71,4.69,9.21
2,8.47,4.24,14.59,42.95,235.38,81.69,0.82,8.31,14.59,4.71,6.14
3,9.78,5.03,13.53,40.59,259.60,78.88,1.28,12.93,7.77,0.00,0.70
4,7.15,4.54,15.55,46.47,180.32,74.15,1.01,8.61,9.21,4.75,6.46


In [27]:
def diabetes_risk(glucose):
    if glucose >= 200:
        return "Critical"
    elif glucose >= 126:
        return "High"
    elif glucose >= 100:
        return "Medium"
    else:
        return "Low"


def anemia_risk(hb):
    if hb < 7:
        return "Critical"
    elif hb < 11:
        return "High"
    elif hb < 13:
        return "Medium"
    else:
        return "Low"


def infection_risk(wbc):
    if wbc > 20:
        return "Critical"
    elif wbc > 11:
        return "High"
    elif wbc < 4:
        return "Low"
    else:
        return "Normal"

In [28]:
bloodReport["diabetes_risk"] = bloodReport["glucose"].apply(diabetes_risk)
bloodReport["anemia_risk"] = bloodReport["hemoglobin"].apply(anemia_risk)
bloodReport["infection_risk"] = bloodReport["wbc"].apply(infection_risk)

bloodReport.head()


,wbc,rbc,hemoglobin,hematocrit,platelets,glucose,creatinine,bun,diabetes_risk,anemia_risk,infection_risk
0,8.25,5.17,16.60,50.58,241.24,90.71,0.52,8.73,Low,Low,Normal
1,7.29,5.56,15.89,47.11,267.77,94.68,0.73,5.40,Low,Low,Normal
2,8.47,4.24,14.59,42.95,235.38,81.69,0.82,8.31,Low,Low,Normal
3,9.78,5.03,13.53,40.59,259.60,78.88,1.28,12.93,Low,Low,Normal
4,7.15,4.54,15.55,46.47,180.32,74.15,1.01,8.61,Low,Low,Normal


In [29]:
LABEL_MAPPING = {
    "Low": "low",
    "Normal": "moderate",
    "Medium": "moderate",
    "High": "high",
    "Critical": "critical"
}

NUM_MAPPING = {
    "low": 0,
    "moderate": 1,
    "high": 2,
    "critical": 3
}

In [30]:
bloodReport["diabetes_risk"] = bloodReport["diabetes_risk"].map(LABEL_MAPPING)
bloodReport["anemia_risk"] = bloodReport["anemia_risk"].map(LABEL_MAPPING)
bloodReport["infection_risk"] = bloodReport["infection_risk"].map(LABEL_MAPPING)
bloodReport["diabetes_risk_num"] = bloodReport["diabetes_risk"].map(NUM_MAPPING)
bloodReport["anemia_risk_num"] = bloodReport["anemia_risk"].map(NUM_MAPPING)
bloodReport["infection_risk_num"] = bloodReport["infection_risk"].map(NUM_MAPPING)
bloodReport.head()

,wbc,rbc,hemoglobin,hematocrit,platelets,glucose,creatinine,bun,diabetes_risk,anemia_risk,infection_risk,diabetes_risk_num,anemia_risk_num,infection_risk_num
0,8.25,5.17,16.60,50.58,241.24,90.71,0.52,8.73,low,low,moderate,0,0,1
1,7.29,5.56,15.89,47.11,267.77,94.68,0.73,5.40,low,low,moderate,0,0,1
2,8.47,4.24,14.59,42.95,235.38,81.69,0.82,8.31,low,low,moderate,0,0,1
3,9.78,5.03,13.53,40.59,259.60,78.88,1.28,12.93,low,low,moderate,0,0,1
4,7.15,4.54,15.55,46.47,180.32,74.15,1.01,8.61,low,low,moderate,0,0,1


In [31]:
"""Preparing data for ML prediction"""
feature_cols = ["wbc","rbc","hemoglobin","hematocrit","platelets","glucose","creatinine","bun"]
X = bloodReport[feature_cols]
y_diabeties = bloodReport["diabetes_risk_num"]
y_anemia = bloodReport["anemia_risk_num"]
y_infection = bloodReport["infection_risk_num"]

In [32]:
"""train test split""" 
X_train,X_test,y_train_diabetes,y_test_diebetes = train_test_split(X, y_diabeties, test_size=0.2, random_state=31, stratify=y_diabeties)
_,_,y_train_anemia,y_test_anemia = train_test_split(X, y_anemia, test_size=0.2, random_state=31, stratify=y_anemia)
_,_,y_train_infection, y_test_infection = train_test_split(X, y_infection, test_size=0.2, random_state=31, stratify=y_infection)

In [33]:
"""Auto detect classes and print report - works for any number of classes"""
CLASS_NAMES = {0: "low", 1: "moderate", 2: "high", 3: "critical"}

def print_report(y_test, y_pred, model_name):
    # automatically finds which classes exist in test + predictions
    existing_labels = sorted(np.unique(np.concatenate([y_test, y_pred])))
    existing_names = [CLASS_NAMES[i] for i in existing_labels]

    print("=" * 40)
    print(f"{model_name} MODEL ACCURACY")
    print("=" * 40)
    print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
    print()
    print("=" * 40)
    print(f"{model_name} CLASSIFICATION REPORT")
    print("=" * 40)
    print(classification_report(
        y_test, y_pred,
        labels=existing_labels,
        target_names=existing_names
    ))

In [34]:
"""Choosing the best hyperparameters for """ 
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
    "gamma": [0, 0.1, 0.3]
}

model = XGBClassifier(
    random_state=31,
    use_label_encoder=False,
    eval_metric="mlogloss"
)

In [35]:
"""Diabeties modal"""
grid_diabeties  = GridSearchCV(
    model,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1 
)
grid_diabeties.fit(X_train, y_train_diabetes)
print("Best Params:", grid_diabeties.best_params_)
print("Best Score:", grid_diabeties.best_score_) 

Best Params: {'colsample_bytree': 0.7, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.7}
Best Score: 0.9974999999999999


In [36]:
model_diabetes = grid_diabeties.best_estimator_
y_predict_diabeties = model_diabetes.predict(X_test)
print_report(y_test_diebetes, y_predict_diabeties, "DIABETIES")

DIABETIES MODEL ACCURACY
Accuracy: 99.00%

DIABETIES CLASSIFICATION REPORT
              precision    recall  f1-score   support

         low       0.99      1.00      0.99        92
    moderate       1.00      0.88      0.93         8

    accuracy                           0.99       100
   macro avg       0.99      0.94      0.96       100
weighted avg       0.99      0.99      0.99       100



In [37]:
"""Anemia Model""" 
grid_anemia = GridSearchCV(
    model,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1 
)
grid_anemia.fit(X_train, y_train_anemia)
print("Best Params:", grid_anemia.best_params_)
print("Best Score:", grid_anemia.best_score_) 

Best Params: {'colsample_bytree': 0.7, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.7}
Best Score: 0.8575000000000002


In [38]:
model_anemia = grid_anemia.best_estimator_
y_predict_anemia = model_anemia.predict(X_test)
print_report(y_test_anemia, y_predict_anemia, "ANEMIA")

ANEMIA MODEL ACCURACY
Accuracy: 86.00%

ANEMIA CLASSIFICATION REPORT
              precision    recall  f1-score   support

         low       0.86      1.00      0.92        86
    moderate       0.00      0.00      0.00        13
        high       0.00      0.00      0.00         1

    accuracy                           0.86       100
   macro avg       0.29      0.33      0.31       100
weighted avg       0.74      0.86      0.80       100



In [39]:
"""Infection Model"""
grid_infection = GridSearchCV(
    model,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1 
)
grid_infection.fit(X_train,y_train_infection)
print("Best Params:", grid_infection.best_params_)
print("Best Score:", grid_infection.best_score_)

Best Params: {'colsample_bytree': 0.7, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.7}
Best Score: 0.9875


In [40]:
model_infection = grid_infection.best_estimator_
y_predict_infection = model_infection.predict(X_test)
print_report(y_test_infection, y_predict_infection, "INFECTION")

INFECTION MODEL ACCURACY
Accuracy: 99.00%

INFECTION CLASSIFICATION REPORT
              precision    recall  f1-score   support

    moderate       0.99      1.00      0.99        99
        high       0.00      0.00      0.00         1

    accuracy                           0.99       100
   macro avg       0.49      0.50      0.50       100
weighted avg       0.98      0.99      0.99       100



In [41]:
"""Saving modals as pkl"""
save_path = r"C:\Users\direk\Disease_risk_predictor_-3\ml_models\xgboost"
os.makedirs(save_path, exist_ok=True)
with open(os.path.join(save_path, "diabeties.pkl"), "wb") as f:
    pickle.dump(model_diabetes, f)
    print("diabetes.pkl saved successfuly!")
with open(os.path.join(save_path, "anemia.pkl"), "wb") as f:
    pickle.dump(model_anemia, f)
    print("anemia.pkl is saved successfully")
with open(os.path.join(save_path, "infection.pkl"), "wb") as f:
    pickle.dump(model_infection, f)
    print("infection.pkl is saved successfully")

diabetes.pkl saved successfuly!
anemia.pkl is saved successfully
infection.pkl is saved successfully
